# SQ1: differential GRN analysis (AD vs control)

Pipeline:
1. Load the SeaAD subsampled inputs + the per-cell wScReNI networks produced by `run_infer_wscreni.ipynb` for this cell type.
2. Tag cells with `condition` (control / ad) and co-pathology covariates.
3. Build the per-donor metadata table (one row per donor, with age / sex / LATE / LBD).
4. Pseudobulk: average each donor's per-cell networks into one (n_genes × n_genes) matrix.
5. For each TF→target candidate edge (from Phase 3 triplets), fit `weight ~ condition + age + sex + LATE + LBD` over donor pseudobulks. Report the condition-coefficient t / p-value with BH-FDR.
6. Output the ranked TF→target table for the supervisor meeting.

Design decisions (see `progress_log.md` 2026-05-09 for rationale):
- AD vs control bin: (Not AD ∪ Low) vs (Intermediate ∪ High) — forced by donor counts.
- Statistical unit: per-donor pseudobulk, not per-cell (avoids pseudoreplication).
- Edge universe: Phase 3 candidate TF→target pairs only (not all gene×gene).
- Test: covariate-adjusted OLS (age confound, mixed co-pathology). Wilcoxon available as sensitivity check.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import anndata as ad

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

from screni.data.combine import combine_wscreni_networks
from screni.data.differential import (
    pseudobulk_per_donor,
    differential_edges,
    ols_with_covariates,
    wilcoxon_unadjusted,
)
from screni.data.loading_seaad import (
    add_condition_column,
    add_copathology_columns,
    select_eligible_donors,
)

In [ ]:
# Configuration — matches the file naming used by the team's Phase 2 + Phase 3
# pipeline for SeaAD: _sub42 suffix (seed embedded), KNN inside the h5ad's
# .uns['knn_indices'], inferred networks under output/networks_seaad_paired/.
CELL_TYPE      = "Microglia-PVM"          # or "L2/3 IT"
DATASET        = "seaad_paired"
SUB_TAG        = "sub42"                  # Phase 2 seed embedded in filenames

DATA_DIR       = os.path.join(REPO_ROOT, "data", "processed", "seaad")
RNA_FULL_PATH  = os.path.join(DATA_DIR, f"{DATASET}_rna.h5ad")                        # full (fallback for obs cols)
RNA_SUB_PATH   = os.path.join(DATA_DIR, f"{DATASET}_rna_{SUB_TAG}.h5ad")              # subsampled (matches inference)
TRIPLETS_PATH  = os.path.join(DATA_DIR, f"{DATASET}_{SUB_TAG}_triplets.csv")

OUT_DIR        = os.path.join(REPO_ROOT, "src", "screni", "data", "output")
NETWORKS_DIR   = os.path.join(OUT_DIR, f"networks_{DATASET}")                         # written by run_infer_wscreni_seaad.ipynb
os.makedirs(OUT_DIR, exist_ok=True)

print(f"cell_type:    {CELL_TYPE}")
print(f"rna_sub:      {RNA_SUB_PATH}")
print(f"triplets:     {TRIPLETS_PATH}")
print(f"networks_dir: {NETWORKS_DIR}")

In [ ]:
# Step 1: load the subsampled RNA (cells × genes) used as input to inference.
# This is the source-of-truth for cell -> donor mapping and for the gene order
# used by combine_wscreni_networks.
rna_sub = ad.read_h5ad(RNA_SUB_PATH)
print(f"rna_sub: {rna_sub.shape}")
print(f"obs cols on rna_sub: {list(rna_sub.obs.columns)}")

# Phase 2 (feature_selection.py) currently strips obs to just 'cell_type' when
# writing the SeaAD _sub.h5ad. We need 'Donor ID', the ADNC column, LATE, LBD,
# Age at Death, Sex. Pull them from the full RNA file (backed mode, no X load)
# and merge into rna_sub.obs by cell barcode.
NEEDED_OBS = [
    "Donor ID", "Overall AD neuropathological Change",
    "LATE", "Highest Lewy Body Disease", "Age at Death", "Sex",
]
missing = [c for c in NEEDED_OBS if c not in rna_sub.obs.columns]
if missing:
    print(f"Backfilling obs columns from full RNA file: {missing}")
    rna_full = ad.read_h5ad(RNA_FULL_PATH, backed="r")
    full_obs = rna_full.obs.loc[rna_sub.obs_names, missing].copy()
    rna_full.file.close()
    for c in missing:
        rna_sub.obs[c] = full_obs[c].values

# Tag condition + co-pathology covariates.
add_condition_column(rna_sub)
add_copathology_columns(rna_sub)

# Drop cells that didn't bin into control/ad (and any with wrong cell type).
mask = (rna_sub.obs["cell_type"].astype(str) == CELL_TYPE) & rna_sub.obs["condition"].notna()
rna_ct = rna_sub[mask].copy()
print(f"after {CELL_TYPE} + condition filter: {rna_ct.shape}")
print(f"  condition: {rna_ct.obs['condition'].value_counts().to_dict()}")
print(f"  donors:    {rna_ct.obs['Donor ID'].nunique()}")

In [ ]:
# Step 2: build the per-donor metadata table that goes into the regression.
donor_meta = select_eligible_donors(
    rna_ct,
    cell_type=CELL_TYPE,
    min_cells_per_donor=1,    # cells already subsampled upstream
    cell_type_col="cell_type",
    donor_col="Donor ID",
    require_condition=True,
).set_index("donor_id")

print(donor_meta)
print(f"\ncontrol: {(donor_meta['condition'] == 'control').sum()}")
print(f"ad:      {(donor_meta['condition'] == 'ad').sum()}")

In [ ]:
# Step 3: load the per-cell wScReNI networks written by run_infer_wscreni.ipynb.
# combine_wscreni_networks reads <network_dir>/wScReNI/<idx>.<cell>.network.txt
# and returns a ScReniNetworks dict keyed by cell name.
networks = combine_wscreni_networks(
    cell_names = rna_ct.obs_names.tolist(),
    network_dir = NETWORKS_DIR,
)
print(f"loaded {len(networks)} per-cell networks")
if networks.gene_names:
    print(f"gene_names[:5]: {networks.gene_names[:5]}")
    first_cell = next(iter(networks))
    print(f"first matrix shape: {networks[first_cell].shape}")

In [ ]:
# Step 4: pseudobulk by donor.
cell_to_donor = rna_ct.obs["Donor ID"].astype(str).to_dict()
pseudobulks = pseudobulk_per_donor(
    networks       = networks,
    cell_to_donor  = cell_to_donor,
    donor_metadata = donor_meta,
    aggregator     = "mean",
)
print(f"pseudobulks: {len(pseudobulks.donor_ids)} donors")
print(pseudobulks.metadata[['condition', 'age', 'sex', 'LATE_present', 'LBD_present', 'n_cells']])

In [ ]:
# Step 5: load the Phase 3 candidate edges and run the differential test.
triplets = pd.read_csv(TRIPLETS_PATH)
print(f"triplets: {triplets.shape}, unique TF->target pairs: "
      f"{triplets[['TF', 'target_gene']].drop_duplicates().shape[0]}")

results = differential_edges(
    pseudobulks = pseudobulks,
    edges       = triplets,
    test        = ols_with_covariates,    # swap to wilcoxon_unadjusted for sensitivity
    cell_type   = CELL_TYPE,
)
print(f"\n{len(results)} edges tested; {int((results['q_value'] < 0.05).sum())} survive q<0.05")
results.head(20)

In [ ]:
# Step 6: save the ranked table for the supervisor meeting.
ct_safe = CELL_TYPE.replace("/", "_").replace(" ", "_").replace("-", "_")
out_path = os.path.join(OUT_DIR, f"differential_edges_{ct_safe}.csv")
results.to_csv(out_path, index=False)
print(f"saved: {out_path}")

## Sensitivity check: Wilcoxon (no covariate adjustment)

If the age / co-pathology covariates carry the signal rather than `condition`, the OLS hits should evaporate when we strip them. This is the same test pattern but using a plain Mann-Whitney U on `ad` vs `control` donors. We expect *partial* overlap with the OLS top edges; large divergence means age or LATE/LBD is doing more work than disease.

In [ ]:
results_wilcox = differential_edges(
    pseudobulks = pseudobulks,
    edges       = triplets,
    test        = wilcoxon_unadjusted,
    cell_type   = CELL_TYPE,
)

# How many of OLS's top-50 also appear in Wilcoxon's top-50?
top_ols = set(zip(results.head(50)['TF'], results.head(50)['target']))
top_wx  = set(zip(results_wilcox.head(50)['TF'], results_wilcox.head(50)['target']))
overlap = top_ols & top_wx
print(f"overlap of top-50 (OLS ∩ Wilcoxon): {len(overlap)} / 50")
results_wilcox.head(10)